# 05 — Customer Sentiment Analysis

Analyze customer sentiment across 12 call-center conversations using RoBERTa and reference-assisted speaker attribution.

**Pipeline:**
1. Restore processed transcripts.
2. Load reference speaker annotations.
3. Extract customer speech.
4. Classify sentiment using RoBERTa.
5. Evaluate sentiment changes and extract evidence.
6. Export results.

## 1. Restore Processed Data

Load the saved summarization dataset containing Whisper transcripts and speaker-aligned conversation turns.

In [11]:
from google.colab import files
from pathlib import Path
import pandas as pd
import json, zipfile

uploaded = files.upload()

with zipfile.ZipFile(next(iter(uploaded))) as z:
    z.extractall("/content")

folder = next(Path("/content").rglob("summarization/input/CALL_001.json")).parent

calls = {
    p.stem: json.loads(p.read_text(encoding="utf-8"))
    for p in sorted(folder.glob("CALL_*.json"))
}

assert len(calls) == 12, f"Expected 12 calls, found {len(calls)}"

print(f"Loaded: {len(calls)}/12 calls")
print("Status: SUCCESS")

Saving Maharah_Summarization_Results_FINAL.zip to Maharah_Summarization_Results_FINAL (2).zip
Loaded: 12/12 calls
Status: SUCCESS


## 2. Load Reference Speaker Annotations

Retrieve ground-truth speaker timestamps and roles from the AppTek Call-Center Dialogues dataset.

Match each selected call using its accent, domain, and duration.

In [12]:
from huggingface_hub import hf_hub_download

records = []

for accent in sorted({c["accent"] for c in calls.values()}):
    path = hf_hub_download(
        repo_id="apptek-com/apptek_callcenter_dialogues",
        repo_type="dataset",
        filename=f"diarization/{accent}/metadata.jsonl"
    )
    records += [json.loads(x) for x in Path(path).read_text().splitlines()]

references = {}

for cid, call in calls.items():
    matches = [
        r for r in records
        if r["accent"] == call["accent"]
        and r["domain"] == call["domain"]
        and abs(r["duration"] - call["duration_sec"]) < 0.15
    ]

    if len(matches) != 1:
        raise ValueError(f"{cid}: Expected one reference, found {len(matches)}")

    references[cid] = matches[0]["segments"]

print(f"Reference annotations loaded: {len(references)}/12")

Reference annotations loaded: 12/12


## 3. Reference-Assisted Speaker Attribution

Align Whisper transcript turns with reference speaker timestamps.

Exclude uncertain speaker assignments to reduce contamination in customer sentiment analysis.

In [13]:
rows = []

for cid, call in calls.items():
    for turn in call["speaker_turns"]:

        if not turn["text"].strip():
            continue

        overlap = {
            role: sum(
                max(0, min(turn["end"], s["end"]) -
                    max(turn["start"], s["start"]))
                for s in references[cid] if s["role"] == role
            )
            for role in ["agent", "customer"]
        }

        role = max(overlap, key=overlap.get)
        total = sum(overlap.values())

        rows.append({
            "call_id": cid,
            "start": turn["start"],
            "text": turn["text"],
            "original_speaker": turn["speaker"],
            "reference_role": role,
            "reference_share": overlap[role] / total if total else 0
        })

turns_df = pd.DataFrame(rows)

customer_df = turns_df[
    (turns_df.reference_role == "customer") &
    (turns_df.reference_share >= 0.70)
].sort_values(["call_id", "start"]).reset_index(drop=True)

display(customer_df.groupby("call_id").size().reset_index(name="customer_turns"))

assert customer_df.call_id.nunique() == 12, "Some calls have no valid customer turns"

print(f"\nCustomer turns extracted: {len(customer_df)}")

,call_id,customer_turns
0,CALL_001,16
1,CALL_002,25
2,CALL_003,33
3,CALL_004,20
4,CALL_005,17
5,CALL_006,9
6,CALL_007,38
7,CALL_008,33
8,CALL_009,24
9,CALL_010,19



Customer turns extracted: 282


## 4. RoBERTa Sentiment Classification

Use a pretrained RoBERTa model to classify customer speech into Positive, Neutral, and Negative sentiment.

Generate class probabilities for each customer turn.

In [14]:
!pip -q install transformers

import torch
from transformers import pipeline

model_name = "cardiffnlp/twitter-roberta-base-sentiment-latest"

classifier = pipeline(
    "text-classification",
    model=model_name,
    top_k=None,
    device=0 if torch.cuda.is_available() else -1
)

predictions = classifier(
    customer_df["text"].tolist(),
    truncation=True,
    batch_size=16
)

scores = pd.DataFrame([
    {p["label"].lower(): p["score"] for p in result}
    for result in predictions
])

customer_df = pd.concat([customer_df, scores], axis=1)

print(f"Classified customer turns: {len(customer_df)}")
display(customer_df.head(5))

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

[transformers] RobertaForSequenceClassification LOAD REPORT from: cardiffnlp/twitter-roberta-base-sentiment-latest
Key                         | Status     |  | 
----------------------------+------------+--+-
roberta.pooler.dense.bias   | UNEXPECTED |  | 
roberta.pooler.dense.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Classified customer turns: 282


,call_id,start,text,original_speaker,reference_role,reference_share,neutral,positive,negative
0,CALL_001,19.74,hi Leo yeah this is Ashley I'm looking into co...,Customer,customer,0.989474,0.713162,0.278920,0.007918
1,CALL_001,39.29,Wow you sound very enthusiastic about this I l...,Customer,customer,1.000000,0.021199,0.975643,0.003158
2,CALL_001,65.57,perfect um I also wanted to know about pricing...,Customer,customer,0.969453,0.557479,0.421590,0.020931
3,CALL_001,99.35,"Wow, all of that sounds amazing.",Customer,customer,1.000000,0.013220,0.979430,0.007350
4,CALL_001,102.97,"Yeah, I think that the canua salad would be a ...",Customer,customer,1.000000,0.194890,0.785418,0.019692


## 5. Call-Level Sentiment Analysis

Aggregate turn-level predictions to identify overall customer sentiment.

Compare opening and closing sentiment and calculate the change in sentiment scores.

In [15]:
labels = ["negative", "neutral", "positive"]
results = []

for cid, g in customer_df.groupby("call_id"):
    n = max(1, min(5, len(g) // 4))

    overall = g[labels].mean()
    opening = g.head(n)[labels].mean()
    closing = g.tail(n)[labels].mean()

    score = lambda x: x["positive"] - x["negative"]

    results.append({
        "call_id": cid,
        "overall_sentiment": overall.idxmax(),
        "confidence": round(overall.max(), 3),
        "opening_sentiment": opening.idxmax(),
        "closing_sentiment": closing.idxmax(),
        "sentiment_change": round(score(closing) - score(opening), 3)
    })

sentiment_df = pd.DataFrame(results)

display(sentiment_df)
print(f"\nProcessed: {len(sentiment_df)}/12 calls")

,call_id,overall_sentiment,confidence,opening_sentiment,closing_sentiment,sentiment_change
0,CALL_001,positive,0.495,positive,neutral,-0.363
1,CALL_002,positive,0.449,neutral,positive,0.843
2,CALL_003,neutral,0.656,neutral,positive,0.629
3,CALL_004,neutral,0.631,neutral,neutral,0.276
4,CALL_005,positive,0.563,neutral,positive,0.447
5,CALL_006,positive,0.547,positive,neutral,-0.510
6,CALL_007,neutral,0.548,neutral,positive,0.611
7,CALL_008,neutral,0.657,neutral,neutral,0.216
8,CALL_009,neutral,0.512,neutral,positive,0.755
9,CALL_010,neutral,0.607,neutral,neutral,0.240



Processed: 12/12 calls


## 6. Sentiment Evidence Extraction

Extract representative customer statements supporting opening and closing sentiment predictions.

Preserve textual evidence for manual validation.

In [16]:
evidence = []

for r in sentiment_df.itertuples():
    g = customer_df[customer_df.call_id == r.call_id]
    n = max(1, min(5, len(g) // 4))

    opening = g.head(n)
    closing = g.tail(n)

    evidence.append({
        "call_id": r.call_id,
        "opening_evidence": opening.loc[
            opening[r.opening_sentiment].idxmax(), "text"
        ][:180],
        "closing_evidence": closing.loc[
            closing[r.closing_sentiment].idxmax(), "text"
        ][:180]
    })

evidence_df = pd.DataFrame(evidence)
final_df = sentiment_df.merge(evidence_df, on="call_id")

display(final_df[[
    "call_id",
    "overall_sentiment",
    "opening_evidence",
    "closing_evidence"
]])

,call_id,overall_sentiment,opening_evidence,closing_evidence
0,CALL_001,positive,"Wow, all of that sounds amazing.",the cinco de mayo
1,CALL_002,positive,My name is Joe Smith. I have a reservation for...,Thank you so much for this is how you keep you...
2,CALL_003,neutral,Erika Black.,"Thanks, Janet. Have a good one."
3,CALL_004,neutral,"Hi, I have a few questions about booking a hot...",Can we do the 10th one and then have a break?
4,CALL_005,positive,"Yeah, so the code number for the clothes and yes","Thank you, I'm very glad to hear that."
5,CALL_006,positive,That's lovely.,"For the complimentary ear mugs, do I have to p..."
6,CALL_007,neutral,I want to know about plants of 5g because,"Yes, yes, I have completed."
7,CALL_008,neutral,"So, hi Rupa Sankar. My name is Bivu. I just bo...","help. So, I think you have sent all the detail..."
8,CALL_009,neutral,I have heard of Mr. Manon. Do you have his abi...,yes yes please confirm okay
9,CALL_010,neutral,"Hi Alison, my name is Jenny and I'm just reque...","I'll just check, just make sure."


## 7. Export Sentiment Results

Save customer-level predictions, call-level sentiment results, and supporting evidence for final integration.

Document the use of reference-assisted speaker attribution.

In [17]:
output = Path("data/processed/sentiment/reference_assisted")
output.mkdir(parents=True, exist_ok=True)

customer_df.to_csv(output / "customer_turn_sentiment.csv", index=False)
final_df.to_csv(output / "call_sentiment_results.csv", index=False)
turns_df.to_csv(output / "speaker_attribution_review.csv", index=False)

metadata = {
    "sentiment_model": model_name,
    "transcript_source": "Whisper ASR",
    "speaker_roles": "AppTek reference-assisted temporal alignment",
    "minimum_reference_share": 0.70,
    "calls_processed": len(final_df)
}

(output / "methodology.json").write_text(
    json.dumps(metadata, indent=2),
    encoding="utf-8"
)

print("STAGE 05 — RESULTS SAVED")
print(f"Calls processed: {len(final_df)}/12")
print(f"Customer turns: {len(customer_df)}")
print(f"Output: {output}")

STAGE 05 — RESULTS SAVED
Calls processed: 12/12
Customer turns: 282
Output: data/processed/sentiment/reference_assisted


## 8. Speaker Attribution Validation

Compare selected Whisper segments with reference transcripts and speaker roles to identify potential attribution errors.

In [18]:
checks = {
    "CALL_001": "the cinco de mayo",
    "CALL_005": "Thank you, I'm very glad to hear that",
    "CALL_006": "That's lovely",
    "CALL_010": "I'll just check"
}

for cid, phrase in checks.items():

    matches = customer_df[
        (customer_df.call_id == cid) &
        customer_df.text.str.contains(phrase, case=False, regex=False)
    ]

    print(f"\n{'='*55}\n{cid}")

    for _, row in matches.iterrows():

        turn = min(
            calls[cid]["speaker_turns"],
            key=lambda t: abs(t["start"] - row["start"])
        )

        ref = max(
            references[cid],
            key=lambda s: max(
                0, min(turn["end"], s["end"]) -
                max(turn["start"], s["start"])
            )
        )

        print("Whisper:", row["text"])
        print("Original role:", row["original_speaker"])
        print("Reference role:", ref["role"])
        print("Reference text:", ref["text"][:250])
        print("Reference share:", round(row["reference_share"], 3))
        print("-"*55)

print("\nValidation inspection completed.")


CALL_001
Whisper: the cinco de mayo
Original role: Customer
Reference role: customer
Reference text: It's Cinco de Mayo.
Reference share: 1.0
-------------------------------------------------------

CALL_005
Whisper: Thank you, I'm very glad to hear that.
Original role: Agent
Reference role: customer
Reference text: Yep, thank you. Thank you, I'm very glad to hear that.
Reference share: 1.0
-------------------------------------------------------

CALL_006
Whisper: That's lovely.
Original role: Agent
Reference role: customer
Reference text: That's lovely. So may I ask, please,
Reference share: 1.0
-------------------------------------------------------

CALL_010
Whisper: I'll just check, just make sure.
Original role: Customer
Reference role: customer
Reference text: All right, I'll just check to make sure.
Reference share: 1.0
-------------------------------------------------------

Validation inspection completed.


## 9. Sentiment Evidence Refinement

Select meaningful customer statements to support sentiment predictions.

Prioritize informative utterances and prepare the results for final validation.

In [19]:
def select_evidence(g, label):
    valid = g[g["text"].str.split().str.len() >= 6]
    if valid.empty:
        valid = g
    return valid.loc[valid[label].idxmax(), "text"]

evidence = []

for r in sentiment_df.itertuples():
    g = customer_df[customer_df.call_id == r.call_id]
    n = max(1, min(5, len(g) // 4))

    evidence.append({
        "call_id": r.call_id,
        "overall_evidence": select_evidence(g, r.overall_sentiment),
        "opening_evidence": select_evidence(g.head(n), r.opening_sentiment),
        "closing_evidence": select_evidence(g.tail(n), r.closing_sentiment)
    })

final_df = sentiment_df.merge(pd.DataFrame(evidence), on="call_id")

pd.set_option("display.max_colwidth", 150)

display(final_df[[
    "call_id", "overall_sentiment",
    "overall_evidence", "closing_evidence"
]])

print(f"\nEvidence refined: {len(final_df)}/12 calls")

,call_id,overall_sentiment,overall_evidence,closing_evidence
0,CALL_001,positive,"Wow, all of that sounds amazing.",yeah of course on so the theme would be um
1,CALL_002,positive,"because I do, I love your facilities are usually very, very,","Thank you so much for this is how you keep your clients. I have to say this. So thank you so much. I do like your facilities, but I do appreciate ..."
2,CALL_003,neutral,and is there a way for us to get like physical tickets or do you guys only have e,"Thanks, Janet. Have a good one."
3,CALL_004,neutral,Would it be possible that the rooms that I have booked that they're adjoining rooms as I'm travelling in a group?,Can we do the 10th one and then have a break?
4,CALL_005,positive,"Thank you, I'm very glad to hear that.","Thank you, I'm very glad to hear that."
5,CALL_006,positive,good morning how may i help you today yes thank you i'm calling in regards to inquiring about purchasing a vip passes for a music festival that's ...,"For the complimentary ear mugs, do I have to pay for it? Was it free of charge?"
6,CALL_007,neutral,"Okay. So, will this be done over the phone call?","No, no, I am helping already. I got all my things done. Thank you."
7,CALL_008,neutral,"and next step, can you tell me that what are the documents do you need to give me this insurance?","help. So, I think you have sent all the details and all the bill over my email."
8,CALL_009,neutral,"Anything I need to bring along like reports, ID?",all right all right i will keep that ready i will keep that ready
9,CALL_010,neutral,I'd be doing it on the website on the,"I'll just check, just make sure."



Evidence refined: 12/12 calls


## 10. Sentiment Quality Control

Exclude an identified mixed-speaker ASR segment and recalculate customer sentiment using existing RoBERTa predictions.

Preserve the original results for comparison.

In [20]:
# Identify mixed-speaker segment
mask = (
    customer_df.call_id.eq("CALL_006") &
    customer_df.text.str.lower().str.startswith(
        "good morning how may i help you today"
    )
)

print("Excluded mixed-speaker turns:", mask.sum())

clean_df = customer_df[~mask].copy()
clean_df.to_csv(output / "customer_turn_sentiment_clean.csv", index=False)

labels = ["negative", "neutral", "positive"]
results = []

for cid, g in clean_df.groupby("call_id"):
    n = max(1, min(5, len(g) // 4))
    overall, opening, closing = [
        x[labels].mean() for x in (g, g.head(n), g.tail(n))
    ]

    score = lambda x: x["positive"] - x["negative"]

    results.append({
        "call_id": cid,
        "overall_sentiment": overall.idxmax(),
        "confidence": round(overall.max(), 3),
        "opening_sentiment": opening.idxmax(),
        "closing_sentiment": closing.idxmax(),
        "sentiment_change": round(score(closing) - score(opening), 3)
    })

sentiment_df = pd.DataFrame(results)

display(sentiment_df)
print(f"\nProcessed: {len(sentiment_df)}/12 calls")

Excluded mixed-speaker turns: 1


,call_id,overall_sentiment,confidence,opening_sentiment,closing_sentiment,sentiment_change
0,CALL_001,positive,0.495,positive,neutral,-0.363
1,CALL_002,positive,0.449,neutral,positive,0.843
2,CALL_003,neutral,0.656,neutral,positive,0.629
3,CALL_004,neutral,0.631,neutral,neutral,0.276
4,CALL_005,positive,0.563,neutral,positive,0.447
5,CALL_006,positive,0.504,positive,neutral,-0.120
6,CALL_007,neutral,0.548,neutral,positive,0.611
7,CALL_008,neutral,0.657,neutral,neutral,0.216
8,CALL_009,neutral,0.512,neutral,positive,0.755
9,CALL_010,neutral,0.607,neutral,neutral,0.240



Processed: 12/12 calls


## 11. Sentiment Justifications

Generate evidence-based explanations for each call's sentiment classification.

Use customer statements and model scores to support predictions and identify cases requiring manual review.

In [21]:
def get_evidence(g, label):
    valid = g[g.text.str.split().str.len().between(6, 70)]
    if valid.empty:
        valid = g

    other = [x for x in labels if x != label]
    margin = valid[label] - valid[other].max(axis=1)

    return valid.loc[margin.idxmax(), "text"].strip()

records = []

for r in sentiment_df.itertuples():
    g = clean_df[clean_df.call_id == r.call_id].sort_values("start")
    n = max(1, min(5, len(g) // 4))

    evidence = get_evidence(g, r.overall_sentiment)

    records.append({
        "call_id": r.call_id,
        "overall_evidence": evidence,
        "opening_evidence": get_evidence(g.head(n), r.opening_sentiment),
        "closing_evidence": get_evidence(g.tail(n), r.closing_sentiment),
        "justification": (
            f"The highest average model score was {r.overall_sentiment} "
            f"({r.confidence:.2f}). Supporting customer statement: "
            f"\"{evidence[:180]}\""
        )
    })

final_df = sentiment_df.merge(pd.DataFrame(records), on="call_id")

final_df.to_csv(output / "sentiment_review_draft.csv", index=False)

pd.set_option("display.max_colwidth", 200)
display(final_df[[
    "call_id", "overall_sentiment", "overall_evidence", "justification"
]])

print(f"\nJustifications generated: {len(final_df)}/12")

,call_id,overall_sentiment,overall_evidence,justification
0,CALL_001,positive,"Wow, all of that sounds amazing.","The highest average model score was positive (0.49). Supporting customer statement: ""Wow, all of that sounds amazing."""
1,CALL_002,positive,"because I do, I love your facilities are usually very, very,","The highest average model score was positive (0.45). Supporting customer statement: ""because I do, I love your facilities are usually very, very,"""
2,CALL_003,neutral,and is there a way for us to get like physical tickets or do you guys only have e,"The highest average model score was neutral (0.66). Supporting customer statement: ""and is there a way for us to get like physical tickets or do you guys only have e"""
3,CALL_004,neutral,Would it be possible that the rooms that I have booked that they're adjoining rooms as I'm travelling in a group?,"The highest average model score was neutral (0.63). Supporting customer statement: ""Would it be possible that the rooms that I have booked that they're adjoining rooms as I'm travelling in a group?"""
4,CALL_005,positive,"Thank you, I'm very glad to hear that.","The highest average model score was positive (0.56). Supporting customer statement: ""Thank you, I'm very glad to hear that."""
5,CALL_006,positive,"Okay, that's fine. Thank you. I think I will go ahead and purchase that now, please.","The highest average model score was positive (0.50). Supporting customer statement: ""Okay, that's fine. Thank you. I think I will go ahead and purchase that now, please."""
6,CALL_007,neutral,"Okay. So, will this be done over the phone call?","The highest average model score was neutral (0.55). Supporting customer statement: ""Okay. So, will this be done over the phone call?"""
7,CALL_008,neutral,"and next step, can you tell me that what are the documents do you need to give me this insurance?","The highest average model score was neutral (0.66). Supporting customer statement: ""and next step, can you tell me that what are the documents do you need to give me this insurance?"""
8,CALL_009,neutral,"Anything I need to bring along like reports, ID?","The highest average model score was neutral (0.51). Supporting customer statement: ""Anything I need to bring along like reports, ID?"""
9,CALL_010,neutral,I'd be doing it on the website on the,"The highest average model score was neutral (0.61). Supporting customer statement: ""I'd be doing it on the website on the"""



Justifications generated: 12/12


## 12. Final Sentiment Justifications

Refine supporting evidence and provide concise, context-based explanations for all 12 calls.

Preserve model predictions and document sentiment limitations.

In [22]:
# Improve selected evidence
fixes = {
    "CALL_002": "this is how you keep your clients",
    "CALL_010": "I'll just check"
}

for cid, phrase in fixes.items():
    matches = clean_df[
        (clean_df.call_id == cid) &
        clean_df.text.str.contains(phrase, case=False, regex=False)
    ]
    if len(matches):
        final_df.loc[final_df.call_id == cid, "overall_evidence"] = matches.iloc[0]["text"]

# Context-based explanations
reasons = {
    "CALL_001": "The customer expresses enthusiasm and explicitly praises the catering options.",
    "CALL_002": "The customer praises the facilities and expresses appreciation for the service.",
    "CALL_003": "The customer mainly asks informational questions about ticket availability.",
    "CALL_004": "The customer discusses hotel booking arrangements without strong emotional language.",
    "CALL_005": "The customer explicitly expresses gratitude and satisfaction with the information received.",
    "CALL_006": "The customer responds positively, thanks the agent, and agrees to proceed with the purchase.",
    "CALL_007": "The customer primarily asks procedural questions about the telecom service.",
    "CALL_008": "The customer requests insurance information and required documentation.",
    "CALL_009": "The customer asks practical questions about appointment requirements.",
    "CALL_010": "The customer discusses website procedures without expressing strong emotions.",
    "CALL_011": "The customer focuses on insurance coverage and claim procedures.",
    "CALL_012": "The customer discusses event scheduling and logistical arrangements."
}

final_df["justification"] = final_df.call_id.map(reasons)
final_df["limitations"] = (
    "Reference-assisted speaker attribution; possible ASR and sentiment classification errors."
)

display(final_df[["call_id", "overall_sentiment", "justification"]])
print(f"\nFinal justifications: {len(final_df)}/12")

,call_id,overall_sentiment,justification
0,CALL_001,positive,The customer expresses enthusiasm and explicitly praises the catering options.
1,CALL_002,positive,The customer praises the facilities and expresses appreciation for the service.
2,CALL_003,neutral,The customer mainly asks informational questions about ticket availability.
3,CALL_004,neutral,The customer discusses hotel booking arrangements without strong emotional language.
4,CALL_005,positive,The customer explicitly expresses gratitude and satisfaction with the information received.
5,CALL_006,positive,"The customer responds positively, thanks the agent, and agrees to proceed with the purchase."
6,CALL_007,neutral,The customer primarily asks procedural questions about the telecom service.
7,CALL_008,neutral,The customer requests insurance information and required documentation.
8,CALL_009,neutral,The customer asks practical questions about appointment requirements.
9,CALL_010,neutral,The customer discusses website procedures without expressing strong emotions.



Final justifications: 12/12


## 13. Final Results Export

Save the validated sentiment analysis outputs, supporting evidence, and methodology for integration into the final application.

In [23]:
import json

output.mkdir(parents=True, exist_ok=True)

clean_df.to_csv(output / "customer_turn_sentiment.csv", index=False)
final_df.to_csv(output / "call_sentiment_results.csv", index=False)
turns_df.to_csv(output / "speaker_attribution_review.csv", index=False)

methodology = {
    "sentiment_model": model_name,
    "transcription": "Whisper Small.en",
    "speaker_attribution": "AppTek reference-assisted alignment",
    "minimum_reference_share": 0.70,
    "excluded_mixed_turns": 1,
    "manual_checks": 4,
    "calls_processed": len(final_df),
    "limitations": [
        "Reference labels are required for speaker attribution.",
        "Some Whisper segments may contain mixed speech.",
        "RoBERTa was not trained specifically on call-center conversations.",
        "Manual checks do not establish overall classification accuracy."
    ]
}

(output / "methodology.json").write_text(
    json.dumps(methodology, indent=2)
)

print("STAGE 05 — COMPLETED")
print(f"Calls: {len(final_df)}/12")
print(f"Customer turns: {len(clean_df)}")
print("\nSentiment distribution:")
print(final_df.overall_sentiment.value_counts())
print("\nResults saved successfully.")

STAGE 05 — COMPLETED
Calls: 12/12
Customer turns: 281

Sentiment distribution:
overall_sentiment
neutral     8
positive    4
Name: count, dtype: int64

Results saved successfully.


## 14. Download Final Results

Package the final sentiment analysis outputs into a ZIP archive for GitHub submission and Streamlit integration.

In [24]:
import shutil

archive = shutil.make_archive(
    "/content/Maharah_Sentiment_Results_FINAL",
    "zip",
    root_dir="/content",
    base_dir="data/processed/sentiment/reference_assisted"
)

files.download(archive)
print("Stage 05 archive created successfully.")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Stage 05 archive created successfully.
